# Use Swapped LogisticRegression for LinearSVC to Evaluate

In [12]:
import pandas as pd
import numpy as np
import random
from sklearn.metrics import cohen_kappa_score
from textblob import TextBlob 
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC 
import time

eval_data = pd.read_csv('eval_Q4.csv')
rest_of_data = pd.read_csv('rest_of_data.csv')

# ==========================================
# 2. INTER-ANNOTATOR AGREEMENT (IAA)
# ==========================================
print("\n--- INTER-ANNOTATOR AGREEMENT ---")
kappa_subj = cohen_kappa_score(eval_data['A1_subj'], eval_data['A2_subj'])
kappa_pol = cohen_kappa_score(eval_data['A1_pol'], eval_data['A2_pol'])

print(f"Subjectivity Cohen's Kappa: {kappa_subj:.2f}")
print(f"Polarity Cohen's Kappa: {kappa_pol:.2f}")

# Use Annotator 1's labels as the "Ground Truth" for training/evaluating
y_true_subj = eval_data['A1_subj']
y_true_pol = eval_data[eval_data['A1_subj'] == 1]['A1_pol']


# ==========================================
# 3. MODEL TRAINING & SPLITTING (ADVANCED SVM)
# ==========================================
print("\n--- TRAINING MODELS ---")

# Ensure balanced splits
X_train, X_test, y_train_subj, y_test_subj = train_test_split(
    eval_data['text'].astype(str), 
    y_true_subj, 
    test_size=0.2, 
    random_state=42,
    stratify=y_true_subj 
)

# UPGRADE 1 & 2: Added stop_words and min_df to remove noise
# UPGRADE 3: Swapped LogisticRegression for LinearSVC
subj_classifier = make_pipeline(
    TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english', min_df=2), 
    LinearSVC(class_weight='balanced', max_iter=2000, random_state=42)
)
subj_classifier.fit(X_train, y_train_subj)

# Create splits for the Polarity model
train_opinionated_mask = y_train_subj == 1
X_train_pol = X_train[train_opinionated_mask]
y_train_pol = eval_data.loc[X_train_pol.index, 'A1_pol']

test_opinionated_mask = y_test_subj == 1
X_test_pol = X_test[test_opinionated_mask]
y_test_pol = eval_data.loc[X_test_pol.index, 'A1_pol']

# UPGRADE 1, 2 & 3: Stop words, min_df, and LinearSVC
pol_classifier = make_pipeline(
    TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english', min_df=2), 
    LinearSVC(class_weight='balanced', max_iter=2000, random_state=42)
)
pol_classifier.fit(X_train_pol, y_train_pol)

print("Models trained successfully.")

# ==========================================
# 4. EVALUATION METRICS 
# ==========================================
print("\n--- EVALUATION METRICS (Subjectivity) ---")
# Predict and evaluate ONLY on the unseen testing split
subj_preds = subj_classifier.predict(X_test)
print(classification_report(y_test_subj, subj_preds, target_names=['Neutral', 'Opinionated']))

print("--- EVALUATION METRICS (Polarity) ---")
# Predict and evaluate ONLY on the unseen opinionated testing split
pol_preds = pol_classifier.predict(X_test_pol)
print(classification_report(y_test_pol, pol_preds, target_names=['Negative', 'Positive']))

# ==========================================
# 5. SCALABILITY & RANDOM ACCURACY TEST
# ==========================================
print("\n--- SCALABILITY & SPEED TEST ---")
start_time = time.time()

# Run the pipeline on the remaining 12,000+ unseen rows
rest_subj_preds = subj_classifier.predict(rest_of_data['text'].astype(str))
rest_opinionated_texts = rest_of_data['text'][rest_subj_preds == 1].astype(str)

if len(rest_opinionated_texts) > 0:
    rest_pol_preds = pol_classifier.predict(rest_opinionated_texts)

end_time = time.time()
elapsed_time = end_time - start_time
total_records = len(rest_of_data)
records_per_sec = total_records / elapsed_time if elapsed_time > 0 else 0

print(f"Processed {total_records} unlabelled records in {elapsed_time:.4f} seconds.")
print(f"Speed: {records_per_sec:.2f} records/second.")

print("\n--- RANDOM ACCURACY SPOT-CHECK ---")
# Sample 5 random records from the unseen data to manually check for your report
rest_of_data['predicted_subj'] = rest_subj_preds
sample = rest_of_data.sample(5, random_state=21)

for idx, row in sample.iterrows():
    label = "Opinionated" if row['predicted_subj'] == 1 else "Neutral"
    print(f"Text: '{row['text'][:80]}...' --> Model Predicted: {label}")


--- INTER-ANNOTATOR AGREEMENT ---
Subjectivity Cohen's Kappa: 0.91
Polarity Cohen's Kappa: 0.94

--- TRAINING MODELS ---
Models trained successfully.

--- EVALUATION METRICS (Subjectivity) ---
              precision    recall  f1-score   support

     Neutral       0.50      0.36      0.42        42
 Opinionated       0.84      0.91      0.87       158

    accuracy                           0.79       200
   macro avg       0.67      0.63      0.64       200
weighted avg       0.77      0.79      0.78       200

--- EVALUATION METRICS (Polarity) ---
              precision    recall  f1-score   support

    Negative       0.65      0.61      0.63        56
    Positive       0.79      0.82      0.81       102

    accuracy                           0.75       158
   macro avg       0.72      0.72      0.72       158
weighted avg       0.74      0.75      0.74       158


--- SCALABILITY & SPEED TEST ---
Processed 12055 unlabelled records in 0.9916 seconds.
Speed: 12156.81 records/se

# Use Transformers to Evaluate (Potential Method)

In [ ]:
pip install sentence-transformers

In [ ]:
import pandas as pd
import time
from sklearn.metrics import cohen_kappa_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sentence_transformers import SentenceTransformer 

# ==========================================
# 1. LOAD DATA
# ==========================================
print("--- LOADING DATA ---")
eval_data = pd.read_csv('eval_Q4.csv')
rest_of_data = pd.read_csv('rest_of_data.csv')

# ==========================================
# 2. INTER-ANNOTATOR AGREEMENT (IAA)
# ==========================================
print("\n--- INTER-ANNOTATOR AGREEMENT ---")
kappa_subj = cohen_kappa_score(eval_data['A1_subj'], eval_data['A2_subj'])
kappa_pol = cohen_kappa_score(eval_data['A1_pol'], eval_data['A2_pol'])

print(f"Subjectivity Cohen's Kappa: {kappa_subj:.2f}")
print(f"Polarity Cohen's Kappa: {kappa_pol:.2f}")

# Use Annotator 1's labels as the "Ground Truth" for training/evaluating
y_true_subj = eval_data['A1_subj']
y_true_pol = eval_data[eval_data['A1_subj'] == 1]['A1_pol']

# ==========================================
# 3. MODEL TRAINING: HYBRID TRANSFORMER
# ==========================================
print("\n--- LOADING TRANSFORMER MODEL ---")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

X_train, X_test, y_train_subj, y_test_subj = train_test_split(
    eval_data['text'].astype(str), 
    y_true_subj, 
    test_size=0.2, 
    random_state=42,
    stratify=y_true_subj 
)

print("--- ENCODING TEXT (This takes a few seconds...) ---")
X_train_emb = embedder.encode(X_train.tolist())
X_test_emb = embedder.encode(X_test.tolist())

print("--- TRAINING MODELS ---")
subj_classifier = LinearSVC(class_weight='balanced', max_iter=2000, random_state=42)
subj_classifier.fit(X_train_emb, y_train_subj)

train_opinionated_mask = y_train_subj == 1
X_train_pol_emb = X_train_emb[train_opinionated_mask]
y_train_pol = eval_data.loc[X_train.index[train_opinionated_mask], 'A1_pol']

test_opinionated_mask = y_test_subj == 1
X_test_pol_emb = X_test_emb[test_opinionated_mask]
y_test_pol = eval_data.loc[X_test.index[test_opinionated_mask], 'A1_pol']

pol_classifier = LinearSVC(class_weight='balanced', max_iter=2000, random_state=42)
pol_classifier.fit(X_train_pol_emb, y_train_pol)

print("Models trained successfully.")

# ==========================================
# 4. EVALUATION METRICS 
# ==========================================
print("\n--- EVALUATION METRICS (Subjectivity) ---")
subj_preds = subj_classifier.predict(X_test_emb)
print(classification_report(y_test_subj, subj_preds, target_names=['Neutral', 'Opinionated']))

print("--- EVALUATION METRICS (Polarity) ---")
pol_preds = pol_classifier.predict(X_test_pol_emb)
print(classification_report(y_test_pol, pol_preds, target_names=['Negative', 'Positive']))

# ==========================================
# 5. SCALABILITY & RANDOM ACCURACY TEST
# ==========================================
print("\n--- SCALABILITY & SPEED TEST ---")

# must extract the text and encode it first!
rest_texts = rest_of_data['text'].astype(str).tolist()

start_time = time.time()

# 1. Encode the unseen data (This is the heavy lifting part of deep learning)
rest_emb = embedder.encode(rest_texts)

# 2. Predict Subjectivity
rest_subj_preds = subj_classifier.predict(rest_emb)

# 3. Filter for opinionated embeddings and predict Polarity
opinionated_mask = rest_subj_preds == 1
rest_opinionated_emb = rest_emb[opinionated_mask]

if len(rest_opinionated_emb) > 0:
    rest_pol_preds = pol_classifier.predict(rest_opinionated_emb)

end_time = time.time()
elapsed_time = end_time - start_time
total_records = len(rest_of_data)
records_per_sec = total_records / elapsed_time if elapsed_time > 0 else 0

print(f"Processed {total_records} unlabelled records end-to-end in {elapsed_time:.4f} seconds.")
print(f"Speed: {records_per_sec:.2f} records/second.")

print("\n--- RANDOM ACCURACY SPOT-CHECK ---")
rest_of_data['predicted_subj'] = rest_subj_preds
sample = rest_of_data.sample(5, random_state=22)

for idx, row in sample.iterrows():
    label = "Opinionated" if row['predicted_subj'] == 1 else "Neutral"
    print(f"Text: '{row['text'][:80]}...' --> Model Predicted: {label}")

--- LOADING DATA ---

--- INTER-ANNOTATOR AGREEMENT ---
Subjectivity Cohen's Kappa: 0.91
Polarity Cohen's Kappa: 0.94

--- LOADING TRANSFORMER MODEL ---


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- ENCODING TEXT (This takes a few seconds...) ---
--- TRAINING MODELS ---
Models trained successfully.

--- EVALUATION METRICS (Subjectivity) ---
              precision    recall  f1-score   support

     Neutral       0.45      0.64      0.53        42
 Opinionated       0.89      0.79      0.84       158

    accuracy                           0.76       200
   macro avg       0.67      0.72      0.68       200
weighted avg       0.80      0.76      0.77       200

--- EVALUATION METRICS (Polarity) ---
              precision    recall  f1-score   support

    Negative       0.55      0.73      0.63        56
    Positive       0.82      0.67      0.74       102

    accuracy                           0.69       158
   macro avg       0.68      0.70      0.68       158
weighted avg       0.72      0.69      0.70       158


--- SCALABILITY & SPEED TEST ---
